1. **공공데이터포털(data.go.kr)**에 접속해서 회원가입
1. 검색창에 '대기오염정보' 또는 **'기상청 단기예보'**를 검색
1. [활용신청] 버튼을 누르면 '일반 인증키'가 발급 (보통 즉시 발급, 활성화까지 1~2시간 걸릴 수 있음)

🍋이 파일은 기상청 단기예보

In [ ]:
#🟢기상청 단기예보
import requests
import pandas as pd
from datetime import datetime

# 1. 내 인증키 설정 (Encoding 키 권장) 🔴serviceKey = "여기에_인증키를_붙여넣으세요"
serviceKey = "6114f483bd6368bb21a7995881e36487d6baeae732d0dcdd3c719842dd0d35ed"

# 2. 오늘 날짜와 시간 설정 (API 요구 형식: YYYYMMDD, HH00)
# 기상청 데이터는 정시(00분) 단위로 업데이트되므로 🟢현재 시간의 정시를 구합니다.
now = datetime.now()
base_date = now.strftime("%Y%m%d")
base_time = now.strftime("%H00")

# 3. 데이터를 요청할 주소 (초단기실황조회)
url = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst'

# 4. 요청 파라미터 (서울 시청 인근 좌표: nx=60, ny=127)
#모든 외부 API는 API 문서(또는 API 가이드)를 제공
params = {
    'serviceKey' : serviceKey, #발급받은 개인 키
    'pageNo' : '1', # 데이터를 나누어 받을 때 조정 (예: 1페이지, 
    'numOfRows' : '1000', #1000개 행)
    'dataType' : 'JSON', #🟢p262 JSON으로 하려면 작성. 기본XML
    'base_date' : base_date,
    'base_time' : base_time,
    'nx' : '60',
    'ny' : '127'
}

# 5. 데이터 요청하기 (딕셔너리를 JSON 문서로 포장해서 보냄)
response = requests.get(url, params=params)

# 6. 결과 확인 및 데이터프레임 변환
# Pandas의 🍋DataFrame()은 데이터를 행(row)과 열(column)로 구성된 
# 2차원 표(스프레드시트) 형태로 다루는 핵심 자료구조
if response.status_code == 200:
    data = response.json()
    if data['response']['header']['resultCode'] == '00': # API 요청이 성공했는지 '00'정상
        items = data['response']['body']['items']['item'] #응답 > 바디 > 아이템들 > 아이템 순으로 겹겹이
        df_weather = pd.DataFrame(items) #날씨 정보(온도, 습도 등)가 들어있는 리스트(item)만 따로 뽑아 표로 변환
        
        # 기상청 코드(category)를 🟢한글로 매핑 (T1H: 기온, REH: 습도 등)
        # PTY 강수형태 UUU 동서바람성분(m/s) VVV 남북바람성분(m/s)
        name_map = {
            'PTY': '강수형태',
            'T1H': '기온(℃)',
            'RN1': '1시간 강수량(mm)',
            'REH': '습도(%)',
            'VEC': '풍향(deg)',
            'WSD': '풍속(m/s)'
        }
        # 3️⃣엑셀 저장 위치를 여기로 하냐와

        df_weather['항목명'] = df_weather['category'].map(name_map) #'category'를 기반으로 '항목명' 열을 만든 
        # 2️⃣-1. 윗줄과 달리 'obsrValue'라는 🍋컬럼의 이름 자체를 '실황 값'으로 변경
        df_weather.rename(columns={'obsrValue': '실황 값'}, inplace=True)
        # 2️⃣-2. 'obsrValue'의 데이터를 가져와 '실황 값'이라는 새로운 🍋열을 만듬
        #df_weather['실황 값'] = df_weather['obsrValue']

        print(f"--- {base_date} {base_time} 기준 기상 실황 ---")
        print(df_weather[['항목명', '실황 값']])
        
        # 엑셀 저장 3️⃣여기로 하냐가 다름
        # df_weather.to_excel('weather_report.xlsx', index=False)
        # print("엑셀 파일 저장 완료!")
    else:
        print("API 응답 오류:", data['response']['header']['resultMsg'])
else:
    print("접속 에러:", response.status_code)

엑셀 파일 저장 완료!
--- 20260410 1200 기준 기상 실황 ---
           항목명 실황 값
0         강수형태    1
1        습도(%)   95
2  1시간 강수량(mm)  0.1
3        기온(℃)  9.6
4          NaN  1.2
5      풍향(deg)  202
6          NaN    3
7      풍속(m/s)  3.2
